In [1]:
import pandas as pd
from data_profiling import ProfileReport
import numpy as np

c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")
df.describe()

C:\Users\paulk\AppData\Local\Temp\ipykernel_59728\1620990980.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.362470e+05,227232.000000,227232.000000,227232.000000,227232.000000,134362.000000
mean,6.924200e+08,14.983180,102.212347,2.678738,46.733937,52603.766683
std,2.572975e+08,47.103571,579.594838,4.529209,4.034939,26780.472001
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,5.243353e+08,2.000000,22.000000,0.774255,44.871519,31112.500000
50%,8.427185e+08,4.000000,22.000000,2.412432,47.340547,57160.000000
75%,8.951636e+08,9.000000,100.000000,4.836760,48.865021,76410.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000


In [3]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [4]:
profile = ProfileReport(df, title="Profiling Report")

In [5]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 136 (\x88) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
E

Rapport généré : rapport_data_profiling.html


In [6]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.

In [7]:
df["siren_amenageur"] = (
    df["siren_amenageur"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

Le type de "siren_amenageur" était un float, on l'a converti en str pour pouvoir effectuer du Regex dessus et checker sa validité 

In [8]:
audit_rows = []

# -------------------------
# 1. COMPLETUDE
# -------------------------
for col in df.columns:
    missing_rate = df[col].isna().mean()

    audit_rows.append({
        "dimension": "completude",
        "colonne": col,
        "pct_problemes": round(missing_rate * 100, 2),
        "gravite": "haute" if missing_rate > 0.3 else "moyenne" if missing_rate > 0.1 else "faible"
    })


# -------------------------
# 2. UNICITE (ID station)
# -------------------------
dup_rate = df.duplicated(subset=["id_station_itinerance"]).mean()

audit_rows.append({
    "dimension": "unicite",
    "colonne": "id_station_itinerance",
    "pct_problemes": round(dup_rate * 100, 2),
    "gravite": "haute" if dup_rate > 0.05 else "moyenne" if dup_rate > 0.01 else "faible"
})


# -------------------------
# 3. VALIDITE - SIREN
# -------------------------
siren_invalid = ~df["siren_amenageur"].astype(str).str.match(r"^\d{9}$")
siren_rate = siren_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "siren_amenageur",
    "pct_problemes": round(siren_rate * 100, 2),
    "gravite": "haute" if siren_rate > 0.1 else "moyenne" if siren_rate > 0.02 else "faible"
})


# -------------------------
# 4. COHERENCE GEO
# -------------------------
geo_invalid = ~(
    df["consolidated_latitude"].between(41, 51) &
    df["consolidated_longitude"].between(-5, 10)
)

geo_rate = geo_invalid.mean()

audit_rows.append({
    "dimension": "coherence_geo",
    "colonne": "latitude_longitude",
    "pct_problemes": round(geo_rate * 100, 2),
    "gravite": "haute" if geo_rate > 0.2 else "moyenne" if geo_rate > 0.05 else "faible"
})


# -------------------------
# 5. VALIDITE PUISSANCE
# -------------------------
power_invalid = df["puissance_nominale"] <= 0
power_rate = power_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "puissance_nominale",
    "pct_problemes": round(power_rate * 100, 2),
    "gravite": "haute" if power_rate > 0.1 else "moyenne" if power_rate > 0.02 else "faible"
})


# -------------------------
# RESULTAT FINAL
# -------------------------
audit_df = pd.DataFrame(audit_rows)
audit_df.sort_values("pct_problemes", ascending=False)

,dimension,colonne,pct_problemes,gravite
37,completude,observations,79.53,haute
27,completude,tarification,75.20,haute
52,unicite,id_station_itinerance,71.86,haute
35,completude,num_pdl,48.96,haute
39,completude,cable_t2_attache,46.97,haute
47,completude,consolidated_code_postal,40.87,haute
36,completude,date_mise_en_service,40.61,haute
53,validite,siren_amenageur,40.05,haute
34,completude,raccordement,34.90,haute
16,completude,id_pdc_local,34.62,haute


In [9]:
audit_df.groupby("dimension")["pct_problemes"].mean()

dimension
coherence_geo     0.530000
completude       11.496538
unicite          71.860000
validite         21.185000
Name: pct_problemes, dtype: float64

On voit que la catégorie qui pose le plus de problèmes est l'unicité avec 72% de données non uniques par colonnes en moyenne puis quelques problèmes de complétude, validité et de cohérence_geo

### Partie 2 

In [10]:
df["num_pdl"].isna().sum()

np.int64(111260)

In [11]:
df.sample(10)

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
157837,RossiniEnergy,839265873,info@rossinienergy.com,RossiniEnergy,info@rossinienergy.com,374090105,RossiniEnergy,FRROSE788,NaN,Herlin_Invest,...,7a5bd754-1a30-4edf-bccf-4c8fb9e9fd6a,rossini-energy,2024-10-22T14:57:42.078000+00:00,2.333351,50.364188,62130.0,Herlin-le-Sec,True,True,False
175238,Ville de la Flèche,nan,NaN,Bouygues Energies & Services,support@alizecharge.fr,0805021480,SARTHE IRVE,FRS72E72154001,NaN,LA FLECHE - Boulevard de Montréal,...,5f29737c-5393-46f9-8140-2509992adc7a,alize,2025-03-17T08:58:10.352000+00:00,-0.071584,47.697526,72200.0,La Flèche,True,True,False
190397,DRIVECO,nan,support@driveco.com,DRIVECO Partner Network,support@driveco.com,NaN,DRIVECO,FRSSDPHESSVOLVO674602,NaN,Volvo - HESS - Souffelweyersheim PDL2 - powere...,...,775dd5a9-c0e4-4bb7-8995-f4b5a4148836,driveco,2026-01-30T10:47:39.688000+00:00,7.726805,48.623329,67460.0,Souffelweyersheim,True,True,False
181829,STATIONS-E,835124280,contact@stations-e.com,STATIONS-E,support@stations-e.com,tel:+33-805-03-51-00,Stations-e,FRSE1PSE03DABA,NaN,Dompierre sur Besbre marché couvert,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,3.680416,46.520675,3290.0,Dompierre-sur-Besbre,True,True,False
168427,SDE35,200022705,bea@sde35.fr,Bouygues Energies et Services,serviceclient@ouestcharge.fr,02.52.07.00.06,OUEST CHARGE,FRS35P35238013B1,NaN,RENNES - 1 place honore commereuc,...,cef7fb5c-481e-4c1b-b770-651b1af49010,syndicat-departemental-denergie-dille-et-vilaine,2025-01-13T11:50:45.971000+00:00,-1.679739,48.108482,NaN,Rennes,True,True,False
115695,Plug Inn fast charge,983504002,fastcharge.exploitation@renault.com,Plug Inn fast charge,fastcharge.exploitation@renault.com,tel:+33-1-83-64-35-76,Plug Inn fast charge,FRMFCPMAURIENNE,NaN,SAINT-JEAN-DE-MAURIENNE,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,6.363461,45.269978,73300.0,Saint-Jean-de-Maurienne,True,True,False
130466,Power Dot France,891118473,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,tel:+33-1-76-31-06-84,Power Dot France,FRPD1PETXFYT,NaN,Kiabi - Saint-Quentin,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,3.255130,49.859137,2100.0,Fayet,True,True,False
87878,IZIVIA FAST,951478437,lucas.malacarne@izivia.com,IZIVIA,sav@izivia.com,+33(0)972668001,IZIVIA FAST,FRIZFPFAST203,FRSODPFAST2031,IZIVIA FAST - McDonald's - Courtabœuf,...,7696ac9e-2789-4eb6-9091-8a9b44cf0721,izivia,2023-11-30T09:39:27.928000+00:00,2.205565,48.683529,91140.0,Villejust,True,True,False
78890,Idex,921041711,leo.marty@idex.fr,Idex,leo.marty@idex.fr,tel:+352-25-36-36-404,Looping,FRIDXP91355366,91355366,Parking Zoo de la Flèche,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,-0.045940,47.677576,NaN,NaN,False,False,False
52972,E-TOTEM,nan,contact@e-totem.fr,E-TOTEM,contact@e-totem.fr,NaN,Réseau e-Totem Infrastructures,FRETIP59172A,FRETIP59172A,e-Totem - Jet Car Wash Denain,...,53f9d331-b845-453c-977d-cd6c48e5bb52,e-totem,2025-06-17T11:55:33.909000+00:00,3.383756,50.322463,59220.0,Denain,True,True,False


In [12]:
df['siren_amenageur']

0               nan
1               nan
2               nan
3               nan
4               nan
            ...    
227227    895193019
227228    508013281
227229    444811970
227230    444811970
227231    444811970
Name: siren_amenageur, Length: 227232, dtype: object

In [13]:
columnes_a_verifier = [
    col for col in df.columns
    if "id_pdc" in col.lower()
    or "itiner" in col.lower()
    or "station" in col.lower()
    or "date" in col.lower()
    or "operateur" in col.lower()
]

print("Colonnes candidates:")
print(columnes_a_verifier)

col_id = "id_pdc_itinerance"
print("\nNombre de lignes:", len(df))
print("Valeurs distinctes de id_pdc_itinerance:", df[col_id].nunique(dropna=False))
print("Lignes en doublon sur id_pdc_itinerance:", df.duplicated(subset=[col_id]).sum())

freq_id = df[col_id].value_counts(dropna=False)
print("\nTop 10 des valeurs les plus fréquentes:")
print(freq_id.head(10))

if len(columnes_a_verifier) > 1:
    print("\nTest de granularité sur plusieurs colonnes:")
    for taille in range(2, min(5, len(columnes_a_verifier)) + 1):
        subset = columnes_a_verifier[:taille]
        doublons = df.duplicated(subset=subset).sum()
        print(f"{subset} -> doublons: {doublons}")

Colonnes candidates:
['nom_operateur', 'contact_operateur', 'telephone_operateur', 'id_station_itinerance', 'id_station_local', 'nom_station', 'implantation_station', 'adresse_station', 'id_pdc_itinerance', 'id_pdc_local', 'station_deux_roues', 'date_mise_en_service', 'date_maj', 'consolidated_longitude', 'consolidated_latitude', 'consolidated_code_postal', 'consolidated_commune', 'consolidated_is_lon_lat_correct', 'consolidated_is_code_insee_verified', 'consolidated_is_code_insee_modified']

Nombre de lignes: 227232
Valeurs distinctes de id_pdc_itinerance: 160138
Lignes en doublon sur id_pdc_itinerance: 67094

Top 10 des valeurs les plus fréquentes:
id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRCG0E002113         3
FRCG0E002114         3
FRCG0E002115         3
FRCG0E002116         3
FRCG0E001353         3
FRALLEGO9004771      3
FRALLEGO9004772      3
FRALLEGO9004773      3
Name: count, dtype: int64

Test de granularité sur plusieurs colonnes:
['nom_operateur', 'con

In [14]:
id_exemple = freq_id.index[1]
exemple = df.loc[df[col_id] == id_exemple, [
    col for col in [
        "id_station_itinerance",
        "id_pdc_itinerance",
        "id_pdc_local",
        "nom_station",
        "nom_operateur",
        "date_mise_en_service",
        "date_maj",
        "consolidated_commune",
        "consolidated_code_postal",
    ] if col in df.columns
]]
print(f"Exemple pour {id_exemple}:")
display(exemple)

Exemple pour FRLIBE3126085:


,id_station_itinerance,id_pdc_itinerance,id_pdc_local,nom_station,nom_operateur,date_mise_en_service,date_maj,consolidated_commune,consolidated_code_postal
105425,FRLIBE3126085,FRLIBE3126085,frlibe3126085,Résidence Carouge,Bornevo,2023-06-23,2023-10-21,Brétigny-sur-Orge,91220.0
105426,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,NaN,2023-11-10,2023-11-10,NaN,NaN
105427,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0
105428,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0


id_pdc_itinerance n'est pas une clef car une ligne représente un snapshot de l'évolution de la mission d'un opérateur. Donc une itinérance peut apparaitre pour plusieurs dates différentes, nom d'opérateur, point de charge local.

Ce qui nous interesse nous, c'est d'avoir une ligne par point de livraison x point de charge

In [15]:
dup_rate = df.duplicated(subset=["id_pdc_itinerance", "num_pdl"]).mean()

print("Taux de doublons PDC×PDL :", round(dup_rate * 100, 2), "%")

Taux de doublons PDC×PDL : 12.43 %


On a tout de même 12.43% de doublons c'est à dire de lignes qui ont le même couple (PDL,PDC)
Pour être sûr de garder un dataset propre, on ne va garder dans les doublons que ceux de la dernière date de mise à jour

In [16]:
df["date_maj"] = pd.to_datetime(df["date_maj"], errors="coerce")

In [44]:
df_clean = (
    df.sort_values("date_maj", ascending=False)
      .drop_duplicates(
          subset=["id_pdc_itinerance", "num_pdl"],
          keep="first"
      )
)

In [43]:
nb_doublons = df_clean.duplicated(
    subset=["id_pdc_itinerance", "num_pdl"]
).sum()

print(f"Nombre de doublons : {nb_doublons}")

Nombre de doublons : 0


On a 160 138 bornes

# PARTIE 3

In [20]:
def fix_encoding(x):
    if isinstance(x, str):
        return (
            x.encode("latin1", errors="ignore")
             .decode("utf-8", errors="ignore")
        )
    return x

df_clean = df_clean.applymap(fix_encoding)

C:\Users\paulk\AppData\Local\Temp\ipykernel_59728\3559430724.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_clean = df_clean.applymap(fix_encoding)


In [21]:
df_clean["consolidated_commune"] = (
    df_clean["consolidated_commune"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.title()
)

In [22]:
df_clean["date_maj"] = pd.to_datetime(df_clean["date_maj"], errors="coerce")
df_clean["date_mise_en_service"] = pd.to_datetime(df_clean["date_mise_en_service"], errors="coerce")
df_clean["created_at"] = pd.to_datetime(df_clean["created_at"], errors="coerce")
df_clean["last_modified"] = pd.to_datetime(df_clean["last_modified"], errors="coerce")

In [23]:
df_clean["puissance_nominale"] = pd.to_numeric(
    df_clean["puissance_nominale"],
    errors="coerce"
)

In [24]:
mask_watts = df_clean["puissance_nominale"] > 1000

df_clean.loc[mask_watts, "puissance_nominale"] /= 1000

In [25]:
df_clean["puissance_nominale"].describe()

count    198977.000000
mean         72.797102
std         101.782091
min           0.000000
25%          22.000000
50%          22.000000
75%         100.000000
max         600.000000
Name: puissance_nominale, dtype: float64

# PARTIE 4

In [30]:
import great_expectations as gx

1. "id_pdc_itinerance" doit être non nulle
>Sans point de charge la ligne n'est plus exploitable

2. "id_station_itinerance" non nulle 
>Un PDC a forcément une station

3. Le subset ["id_pdc_itinerance","num_pdl"] doit être unique
> On prend toujours la dernière date de MàJ

4. "puissance_nominale" est > 0 et <600kW
> Une borne standard fournie entre 0 et 400kW, exceptionnellement 500kW donc 600 permet d'éliminer les valeurs aberrantes 

5. "consolidated_latitude" entre 41 et 52 

6. consolidated_longitude entre -5.5 et 10
>Le dataset concerne les IRVE françaises donc on adapte les coordonnées en fonction

7. Le code postal doit être une suite de 5 chiffres 
>Standard des codes FR

8. Le siren doit être une suite de 9chiffres
>Standard Français

In [34]:
print(gx.__version__)

1.18.2


In [46]:
import great_expectations as gx

context = gx.get_context()

datasource = context.data_sources.add_pandas(
    name="pandas_source"
)

asset = datasource.add_dataframe_asset(
    name="irve_asset"
)

batch_definition = asset.add_batch_definition_whole_dataframe(
    "whole_dataframe"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_clean}
)

In [47]:
expectation = gx.expectations.ExpectCompoundColumnsToBeUnique(
    column_list=["id_pdc_itinerance", "num_pdl"]
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 9/9 [00:00<00:00,  9.39it/s]  

True


In [48]:
expectation = gx.expectations.ExpectCompoundColumnsToBeUnique(
    column_list=["id_pdc_itinerance", "num_pdl"]
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 9/9 [00:01<00:00,  7.44it/s]  

True


In [51]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="consolidated_latitude",
    min_value=41,
    max_value=52
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 15.24it/s] 

False


In [52]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="consolidated_longitude",
    min_value=-5.5,
    max_value=10
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 16.14it/s] 

False


In [53]:
expectation = gx.expectations.ExpectColumnValuesToMatchRegex(
    column="siren_amenageur",
    regex=r"^\d{9}$"
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 10.37it/s] 

False


In [56]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="puissance_nominale",
    min_value=0,
    max_value=600
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 13.42it/s] 

False
